# Lesson 7 Practice — Feature Transformation
## Modules 1–3: Student Version

**Course:** Data Preparation & Visualization  
**Goal:** For each exercise, **calculate by hand first**, write your result, and only then use Python/scikit-learn to check it.

### Topics
1. Numerical Feature Scaling
   - MinMaxScaler
   - StandardScaler
   - RobustScaler
   - Why scaling matters for distance
   - Train/test leakage
2. Categorical Feature Encoding
   - OrdinalEncoder
   - OneHotEncoder
   - Dummy-variable trap
   - Unknown categories
   - Target encoding by hand
3. Distribution Transformations
   - Box-Cox
   - Yeo-Johnson
   - QuantileTransformer: uniform and normal

> All numerical examples in this notebook are newly created for practice and are different from the toy examples used in the lecture slides.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler,
    OrdinalEncoder, OneHotEncoder,
    PowerTransformer, QuantileTransformer
)
from sklearn.model_selection import train_test_split

np.set_printoptions(suppress=True, precision=4)
pd.set_option('display.precision', 4)

# Module 1 — Numerical Feature Scaling

## Exercise 1 — Why does scaling matter for distance?

Three stores are described by **annual revenue** (thousand USD) and **customer rating**:

| Store | Revenue | Rating |
|---|---:|---:|
| A | 120 | 4.2 |
| B | 180 | 4.4 |
| C | 300 | 3.8 |

### Hand calculation
1. Calculate the Euclidean distance between A and B using the raw variables.
2. Calculate the Euclidean distance between A and C using the raw variables.
3. Which variable dominates the distance?
4. Standardize both variables using the **population** standard deviation:
$$
z=\frac{x-\mu}{\sigma}
$$
5. Recalculate the two Euclidean distances after standardization.
6. Explain why a distance-based algorithm may behave differently after scaling.

In [ ]:
stores = pd.DataFrame({
    'Revenue': [120, 180, 300],
    'Rating': [4.2, 4.4, 3.8]
}, index=['A', 'B', 'C'])

# TODO 1: display the data
# YOUR CODE HERE

# TODO 2: calculate raw Euclidean distances A-B and A-C
# YOUR CODE HERE

# TODO 3: standardize the two columns with StandardScaler
# YOUR CODE HERE

# TODO 4: calculate Euclidean distances again
# YOUR CODE HERE

## Exercise 2 — Min-Max scaling by hand

Daily advertising expenditure (hundred USD):

$$
X=[8,\ 12,\ 18,\ 23,\ 29]
$$

Use:

$$
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
$$

### Hand calculation
1. Find $x_{\min}$ and $x_{\max}$.
2. Transform **12**, **18**, and **29** manually.
3. Transform all five observations.
4. What are the minimum and maximum of the transformed data?
5. Predict what happens if a new value $x=35$ is transformed using the **same fitted scaler**.

In [ ]:
X_minmax = np.array([8, 12, 18, 23, 29]).reshape(-1, 1)

# TODO: fit MinMaxScaler and compare with your hand calculation
# YOUR CODE HERE

# TODO: transform the new value 35 using the already-fitted scaler
# YOUR CODE HERE

## Exercise 3 — StandardScaler by hand

Weekly order volumes:

$$
X=[42,\ 48,\ 54,\ 60,\ 66]
$$

Scikit-learn's `StandardScaler` uses the population variance:

$$
\mu=\frac{1}{N}\sum x_i,\qquad
\sigma=\sqrt{\frac{1}{N}\sum(x_i-\mu)^2}
$$

and

$$
z_i=\frac{x_i-\mu}{\sigma}
$$

### Hand calculation
1. Calculate the mean.
2. Calculate the population variance and standard deviation.
3. Find the z-scores of 42, 54, and 66.
4. What should the mean of the transformed values be?
5. What should their population variance be?

In [ ]:
X_std = np.array([42, 48, 54, 60, 66]).reshape(-1, 1)

# TODO: use StandardScaler
# Print mean_, var_, scale_, and transformed values.
# YOUR CODE HERE

## Exercise 4 — RobustScaler and an extreme outlier

Transaction amounts:

$$
X=[15,\ 18,\ 21,\ 24,\ 27,\ 30,\ 150]
$$

Use the lecture convention:

$$
x_{\text{robust}}=\frac{x-\text{Median}}{Q_3-Q_1}
$$

For this exercise, use:
- Median = 24
- $Q_1=18$
- $Q_3=30$

### Hand calculation
1. Calculate the IQR.
2. Transform 15, 24, 30, and 150 manually.
3. Now calculate the StandardScaler z-score of 150.
4. Compare what StandardScaler and RobustScaler do to the central observations.
5. Which scaler's **center and scale estimates** are less affected by the outlier?

In [ ]:
X_rob = np.array([15, 18, 21, 24, 27, 30, 150]).reshape(-1, 1)

# TODO 1: RobustScaler
# YOUR CODE HERE

# TODO 2: StandardScaler
# YOUR CODE HERE

# TODO 3: put raw, robust-scaled, and standard-scaled values in one DataFrame
# YOUR CODE HERE

## Exercise 5 — Data leakage: fit on train, transform test

Suppose:

$$
X_{\text{train}}=[10,\ 20,\ 30,\ 40]
$$

and

$$
X_{\text{test}}=[50,\ 70]
$$

### Hand calculation
1. Fit a Min-Max transformation using **training data only**.
2. Transform 50 and 70 using the training minimum and maximum.
3. Can a transformed test value be larger than 1? Explain.
4. Now imagine fitting the scaler on train + test together. Why is this data leakage?

In [ ]:
X_train = np.array([10, 20, 30, 40]).reshape(-1, 1)
X_test  = np.array([50, 70]).reshape(-1, 1)

# Correct workflow
scaler = MinMaxScaler()

# TODO: fit ONLY on X_train, then transform both sets
# YOUR CODE HERE

# Incorrect workflow for comparison only:
# TODO: fit on the combined data and compare the transformed training values
# YOUR CODE HERE

# Module 2 — Categorical Feature Encoding

## Exercise 6 — Ordinal encoding

A customer survey records service satisfaction:

$$
\text{Poor} < \text{Fair} < \text{Good} < \text{Very Good} < \text{Excellent}
$$

Observed data:

`Good, Poor, Excellent, Fair, Very Good, Good`

### Hand calculation
1. Create a sensible integer mapping starting at 0.
2. Encode the six observations manually.
3. Why is `OrdinalEncoder` appropriate here?
4. Would it be appropriate for city names such as Hanoi, Hue, and Da Nang? Why?

In [ ]:
satisfaction = pd.DataFrame({
    'Satisfaction': ['Good', 'Poor', 'Excellent', 'Fair', 'Very Good', 'Good']
})

order = [['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']]

# TODO: create and apply OrdinalEncoder using the specified order
# YOUR CODE HERE

## Exercise 7 — One-hot encoding and the dummy-variable trap

Payment method:

`Cash, Card, Transfer, Card, E-wallet, Cash`

### Hand calculation
1. List all unique categories.
2. Construct the one-hot vector for:
   - Cash
   - Card
   - Transfer
   - E-wallet
3. If an intercept is included in a linear regression, explain why using all dummy columns creates perfect multicollinearity.
4. If one category is dropped, how many dummy columns remain?

In [ ]:
payment = pd.DataFrame({
    'Payment': ['Cash', 'Card', 'Transfer', 'Card', 'E-wallet', 'Cash']
})

# TODO 1: OneHotEncoder with all categories
# Hint: sparse_output=False
# YOUR CODE HERE

# TODO 2: repeat with drop='first'
# YOUR CODE HERE

## Exercise 8 — Unknown categories in new data

Training categories:

`North, South, Central, North, South`

New observations:

`North, West`

### Before coding
1. What problem occurs if the encoder has never seen `West`?
2. What do you expect when `handle_unknown='ignore'` is used?
3. What should the one-hot vector for `West` look like?

In [ ]:
train_region = pd.DataFrame({'Region': ['North', 'South', 'Central', 'North', 'South']})
new_region = pd.DataFrame({'Region': ['North', 'West']})

# TODO: use OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# Fit on train_region and transform new_region.
# YOUR CODE HERE

## Exercise 9 — Target encoding by hand

The training data are:

| Industry | Profit Margin (%) |
|---|---:|
| Retail | 8 |
| Tech | 20 |
| Retail | 12 |
| Food | 6 |
| Tech | 24 |
| Food | 10 |
| Tech | 16 |
| Retail | 10 |

### Hand calculation
1. Compute the global mean target.
2. Compute the mean target for Retail, Tech, and Food.
3. Replace each category by its group mean.
4. Why can naive target encoding create target leakage/overfitting?
5. What should be done in a real machine-learning workflow?  
   *Hint: compute encodings using training folds only.*

In [ ]:
df_te = pd.DataFrame({
    'Industry': ['Retail','Tech','Retail','Food','Tech','Food','Tech','Retail'],
    'ProfitMargin': [8,20,12,6,24,10,16,10]
})

# TODO 1: calculate global mean
# YOUR CODE HERE

# TODO 2: calculate category means using groupby
# YOUR CODE HERE

# TODO 3: create a new column using map or transform
# YOUR CODE HERE

# Module 3 — Distribution Transformations

## Exercise 10 — Box-Cox transformation by hand

Positive monthly claim amounts:

$$
X=[1,\ 4,\ 9,\ 16,\ 25]
$$

The Box-Cox transformation is

$$
x^{(\lambda)}=
\begin{cases}
\frac{x^\lambda-1}{\lambda}, & \lambda\neq 0\\
\ln(x), & \lambda=0
\end{cases}
$$

### Hand calculation
Use **fixed** $\lambda=0.5$.

1. Derive the simplified formula for $\lambda=0.5$.
2. Transform all five observations manually.
3. Compare the raw gaps with the transformed gaps.
4. Does the transformation compress large values?
5. Why can Box-Cox not be directly applied to zero or negative observations?

In [ ]:
X_box = np.array([1, 4, 9, 16, 25], dtype=float)

# TODO: implement the Box-Cox formula manually for lambda = 0.5
# Do NOT use PowerTransformer for this first check.
# YOUR CODE HERE

# Optional verification with scipy.stats.boxcox using fixed lambda:
from scipy.stats import boxcox
# YOUR CODE HERE

## Exercise 11 — Yeo-Johnson transformation by hand

Yeo-Johnson can handle positive, zero, and negative values.

Use:

$$
X=[-8,\ -3,\ 0,\ 3,\ 8]
$$

For this exercise, use fixed $\lambda=1$.

### Hand calculation
Use the Yeo-Johnson piecewise formula from the lecture.

1. Transform each observation manually.
2. What happens to positive values when $\lambda=1$?
3. What happens to negative values when $\lambda=1$?
4. Why is Yeo-Johnson more flexible than Box-Cox with respect to the domain of $X$?

In [ ]:
X_yj = np.array([-8, -3, 0, 3, 8], dtype=float)

# TODO: write a small function implementing Yeo-Johnson for lambda=1
# YOUR CODE HERE

# Then compare with scipy.stats.yeojohnson using lmbda=1
from scipy.stats import yeojohnson
# YOUR CODE HERE

## Exercise 12 — Let PowerTransformer estimate lambda

Customer account balances:

$$
X=[-12,\ -4,\ 0,\ 2,\ 7,\ 18,\ 55]
$$

### Before coding
1. Which method can be used: Box-Cox, Yeo-Johnson, or both?
2. Explain your answer using the domain restrictions.
3. Predict whether the large positive value 55 will be compressed after transformation.

### Coding
Fit `PowerTransformer(method='yeo-johnson', standardize=False)` and inspect `lambdas_`.

Then fit it again with `standardize=True`.

Compare the two outputs.

In [ ]:
X_power = np.array([-12, -4, 0, 2, 7, 18, 55], dtype=float).reshape(-1, 1)

# TODO 1: Yeo-Johnson without standardization
# YOUR CODE HERE

# TODO 2: print estimated lambda and transformed values
# YOUR CODE HERE

# TODO 3: repeat with standardize=True
# YOUR CODE HERE

## Exercise 13 — QuantileTransformer: empirical CDF by hand

A new dataset of delivery times is already sorted:

$$
X=[11,\ 13,\ 17,\ 26,\ 41,\ 68,\ 95,\ 140]
$$

For the **hand-calculation exercise**, use the lecture midpoint empirical CDF:

$$
F(x_{(i)})=\frac{i-0.5}{N}
$$

### Hand calculation
1. Here $N=8$. Calculate $F$ for all eight observations.
2. Calculate $F$ for $x=11$, $x=26$, and $x=140$.
3. Explain why subtracting 0.5 avoids exactly 0 and 1.
4. Why is this useful before applying $\Phi^{-1}(F)$?
5. Compare the raw gap $140-95$ with the quantile gap between their midpoint probabilities.

> **Important:** scikit-learn's `QuantileTransformer` uses its own empirical quantile interpolation and boundary convention, so its exact endpoint values are not the same as the midpoint plotting-position formula above. Treat the hand calculation as the lecture method, then use scikit-learn as the practical implementation.

In [ ]:
X_q = np.array([11, 13, 17, 26, 41, 68, 95, 140], dtype=float).reshape(-1, 1)

# TODO 1: calculate the lecture midpoint probabilities manually in Python
# YOUR CODE HERE

# TODO 2: practical sklearn implementation: output_distribution='uniform'
# Set n_quantiles=8
# YOUR CODE HERE

## Exercise 14 — QuantileTransformer: Uniform → Normal

Continue with:

$$
X=[11,\ 13,\ 17,\ 26,\ 41,\ 68,\ 95,\ 140]
$$

Using your midpoint probabilities from Exercise 13:

$$
z=\Phi^{-1}(F)
$$

### Hand calculation
Use a normal table or calculator to approximate:
- $\Phi^{-1}(0.0625)$
- $\Phi^{-1}(0.4375)$
- $\Phi^{-1}(0.9375)$

Then:
1. Match these values to the relevant raw observations.
2. Explain why the transformed values become much more evenly spread.
3. Explain the **distance distortion** trade-off.

After that, use scikit-learn with `output_distribution='normal'` and compare its practical output with the lecture midpoint calculation.

In [ ]:
from scipy.stats import norm

# TODO 1: calculate normal scores from the lecture midpoint probabilities
# YOUR CODE HERE

# TODO 2: sklearn QuantileTransformer with output_distribution='normal'
# YOUR CODE HERE

## Exercise 15 — Final comparison: choose the transformation

For each situation below, choose the most appropriate method **from Modules 1–3** and justify your choice in 1–2 sentences.

1. A KNN model uses `AnnualIncome` (0–200,000) and `SatisfactionScore` (1–5), with no serious outliers.
2. A numerical feature contains several extreme measurement spikes, but preserving the relative spacing of normal observations is important.
3. A feature is strongly right-skewed and contains zero and negative values.
4. A feature has a very irregular/multimodal distribution and severe outliers; rank information is more important than original distances.
5. A categorical variable is `Small < Medium < Large < Very Large`.
6. A categorical variable is `Hanoi, Bangkok, Tokyo, Seoul`.
7. A production model may receive a new city not observed during training.
8. You scale data before model training. Should the scaler be fitted before or after the train/test split?

Write your answers below.

In [ ]:
# Write short answers as comments or Markdown.
# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.

# Optional Challenge — Mini mixed-data workflow

Create the following dataset and prepare it for a distance-based model.

- `Age`: numerical
- `MonthlySpend`: numerical with an outlier
- `Membership`: ordinal (`Bronze < Silver < Gold`)
- `Channel`: nominal (`Store`, `Web`, `App`)

Tasks:
1. Decide how to transform each feature.
2. Fit transformations on a training set only.
3. Transform the test set.
4. Explain every choice.

In [ ]:
df = pd.DataFrame({
    'Age': [21, 25, 31, 38, 45, 52, 29, 34, 41, 60],
    'MonthlySpend': [120, 150, 170, 210, 260, 310, 145, 190, 240, 1800],
    'Membership': ['Bronze','Silver','Silver','Gold','Gold','Gold','Bronze','Silver','Gold','Silver'],
    'Channel': ['Web','Store','App','Web','Store','App','App','Web','Store','Web']
})

# YOUR CODE HERE